# Mimic Logic: a ReAct Puzzle Generator Agent
#### Built with LangChain, following the structure of the course's *ReAct Agent with LangChain* tutorial

In this notebook we build the **Puzzle Generator agent** for Mimic Logic. Instead of following a fixed recipe, a language model **designs each room**: it reads a design brief, decides where the mimics hide, writes the chests' statements, reads the solver's feedback, and revises until the room works.

The model is creative; the **solver keeps it honest**. A room reaches the player only if the solver confirms it has **exactly one answer**.

**Sections**

1. Installs and imports
2. Setup your API key
3. The game data and tools
4. Build the agent with LangChain
5. Generate rooms
6. Evaluate the agent
7. Your turn

## What the agent does

Each run produces one room at a chosen difficulty level (1 to 5):

| Step | ReAct | What happens |
|---|---|---|
| 1 | Act | `get_design_brief`: read this room's random **design brief** |
| 2 | Think | plan the hidden answer and the key deduction |
| 3 | Act | `check_room`: submit a draft; the solver reports broken statements, ambiguity, and difficulty |
| 4 | Think → Act | revise (optionally with `statement_options`) and check again |
| 5 | Act | `submit_room`: accepted only if the last draft has exactly one answer |

The design brief is drawn at random (grid, chest count, mimic count, and a creative **twist** such as *"Two chests accuse each other, and only one of them is right"*), so no two rooms start from the same plan.

# 1. Installs and Imports

In [1]:
%%capture
!pip install langchain langchain-openai openai python-dotenv

This notebook imports the shared Mimic Logic code from two files that must sit **next to the notebook** (in Colab, upload them with the 📁 Files panel):

* `puzzle.py`: the rules, statement types, and the **solver** (shared with the game screen)
* `generator_agent.py`: the algorithmic generator, used here as a **fallback** if the agent fails to deliver a room

`offline_model.py` is optional: a scripted stand-in for the model, for testing without an API key.

In [2]:
# Standard library
import os
import json
import random
from typing import Literal

# Shared Mimic Logic code (must be next to this notebook)
from puzzle import (ARITY, DIRECTIONS, INSTRUCTIONS, PATTERN_RULES, PATTERNS, RULES,
                    RULES_VERSION, STATEMENT_TYPES, check_answer, design_problems, grid_of,
                    is_giveaway, make_room_id, render, solve_all, solve_with_trace,
                    statement_key, statement_text, statement_truth)
from generator_agent import GeneratorAgent

# 2. Setup your API Key

We use **OpenRouter**, exactly as in the tutorial. Store the key as the Colab secret `OPENROUTER_API_KEY`, or in a local `.env` file:

```text
OPENROUTER_API_KEY=your_openrouter_key_here
```

**Treat this key like a password.** Keep `.env` in `.gitignore`.

In [3]:
# Uncomment the lines below if running in Google Colab
# from google.colab import userdata
# os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
# print("API key loaded from Colab Secrets")

In [4]:
# Load the API key from .env file if running locally
from dotenv import load_dotenv
load_dotenv()
print("Environment loaded from .env file")

Environment loaded from .env file


In [5]:
# Set to True to run the notebook WITHOUT an API key, using a scripted stand-in
# (offline_model.py). The stand-in is not a language model: use it only to test the
# harness or as a demo backup.
USE_OFFLINE_MODEL = False

if not USE_OFFLINE_MODEL and not os.getenv("OPENROUTER_API_KEY"):
    raise RuntimeError(
        "Missing OPENROUTER_API_KEY. Check Colab Secrets or your local .env file, "
        "or set USE_OFFLINE_MODEL = True to test without a key."
    )
print("Using the offline stand-in." if USE_OFFLINE_MODEL else "OPENROUTER_API_KEY is available.")

OPENROUTER_API_KEY is available.


## Choose the model

Same model as the tutorial: `openai/gpt-4.1-mini` through OpenRouter, small, inexpensive, and reliable at tool calling.

**Temperature is not 0 here.** The tutorial used 0 so that answers were reproducible. A puzzle generator has the opposite goal: we *want* variety. We use 0.7, so the same brief can lead to different rooms. Variety also comes from the random design brief, which we *can* make reproducible with a seed when we evaluate.

In [6]:
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
MODEL = "openai/gpt-4.1-mini"
TEMPERATURE = 0.7

print(f"Model: {MODEL} (temperature {TEMPERATURE})")

Model: openai/gpt-4.1-mini (temperature 0.7)


# 3. The Game Data and Tools

As in the tutorial, we write the tools as **plain Python functions first** and test them directly. If a tool is wrong, no prompt will make the agent right.

## The game data: levels and design briefs

Each level sets the grid sizes, chest counts, allowed statement types, and the difficulty we aim for. `make_brief` draws a random brief inside those limits and adds a **twist**: a creative constraint for the model to design around.

In [7]:
LEVELS = {
    1: {"grids": ["2x2"], "chests": (3, 3), "types": ["is_mimic", "not_mimic"],
        "show_count": True, "silent": (0, 0), "target_difficulty": (1, 2)},
    2: {"grids": ["2x2", "2x3"], "chests": (4, 4),
        "types": ["is_mimic", "not_mimic", "either_one", "same_kind"],
        "show_count": True, "silent": (0, 0), "target_difficulty": (2, 3)},
    3: {"grids": ["2x3", "3x2"], "chests": (4, 5),
        "types": ["is_mimic", "not_mimic", "either_one", "same_kind", "if_then"],
        "show_count": True, "silent": (0, 1), "target_difficulty": (2, 4)},
    4: {"grids": ["3x3", "2x3"], "chests": (5, 6), "types": list(STATEMENT_TYPES),
        "show_count": True, "silent": (0, 1), "target_difficulty": (3, 5)},
    5: {"grids": ["3x3"], "chests": (5, 7), "types": list(STATEMENT_TYPES),
        "show_count": False, "silent": (1, 1), "target_difficulty": (4, 5)},
}

# (twist, statement type it needs, or None)
TWISTS = [
    ("Two chests accuse each other, and only one of them is right.", "is_mimic"),
    ("The final deduction should come from the mimic-count hint.", None),
    ("Make a corner chest the key to the whole room.", None),
    ("No chest directly calls another chest a mimic.", "either_one"),
    ("A Normal chest should look the most suspicious at first glance.", None),
    ("Use direction words ('the chest below me') for most statements.", None),
    ("Include a conditional whose 'if' part turns out false.", "if_then"),
    ("The mimics should form a recognisable shape on the grid.", "pattern"),
    ("Two chests of the same kind should be far apart on the grid.", "same_kind"),
    ("Let one chest's statement be true but useless on its own.", None),
]

def make_brief(level: int, rng: random.Random) -> dict:
    level = max(1, min(5, int(level)))
    g = LEVELS[level]
    types = list(g["types"])
    n = rng.randint(*g["chests"])
    grids = [gr for gr in g["grids"] if int(gr[0]) * int(gr[2]) >= n] or ["3x3"]
    twists = [t for t, needs in TWISTS if needs is None or needs in types]
    return {
        "level": level,
        "grid": rng.choice(grids),
        "num_chests": n,
        "num_mimics": rng.randint(1, max(1, n // 2)),
        "allowed_types": types,
        "show_mimic_count": g["show_count"],
        "silent_chests": rng.randint(*g["silent"]),
        "target_difficulty": list(g["target_difficulty"]),
        "twist": rng.choice(twists),
    }

make_brief(3, random.Random(7))

{'level': 3,
 'grid': '2x3',
 'num_chests': 5,
 'num_mimics': 2,
 'allowed_types': ['is_mimic',
  'not_mimic',
  'either_one',
  'same_kind',
  'if_then'],
 'show_mimic_count': True,
 'silent_chests': 0,
 'target_difficulty': [2, 4],
 'twist': 'The final deduction should come from the mimic-count hint.'}

**Why level 5 has exactly one silent chest.** It first allowed two. In testing, the agent kept failing level 5 briefs with 7 chests, 2 of them silent, and a hidden mimic count. We then ran the algorithmic generator on the same settings: it often could not find a valid room either, while with one silent chest it succeeded every time. The brief, not the agent, was the problem: five statements and no count leave too little information to pin down seven chests.

## The session

The tools need to share state during one run: the brief and the last draft the solver checked. We keep it in a small `SESSION` dictionary that `start_session` resets before each room.

In [8]:
SESSION = {}

def start_session(level: int, seed: int | None = None) -> dict:
    rng = random.Random(seed)
    SESSION.clear()
    SESSION.update(brief=make_brief(level, rng), rng=rng,
                   draft=None, draft_unique=False, final=None, checks=0)
    return SESSION["brief"]

### Tool 1: `get_design_brief`

Returns this room's design brief: grid, chest count, mimic count, allowed statement types, silent chests, whether the mimic count is shown, the target difficulty, and a creative twist.

In [9]:
def get_design_brief() -> dict:
    return SESSION["brief"]

### Tool 2: `check_room`

The heart of the harness. The model sends a whole draft; the tool:

1. **validates the format** (every problem listed, so the model can fix them all at once);
2. checks each statement against the model's **intended answer** (a Normal chest's statement must be true, a Mimic's false), and for each one that breaks it, **suggests replacements that fit**;
3. checks the draft against the **brief** (grid, chest count, mimic count, silent chests, mimic-count setting);
4. asks the solver **how many answers** the room has, and if more than one, which chests the competing answers disagree on;
5. reports the **difficulty** against the brief's target.

When a room is **ambiguous**, `check_room` also searches every one-statement change (replace a statement, give a silent chest one, or remove one) and reports up to three that make the room uniquely solvable without breaking the brief's silent-chest count, preferring ones that hit the target difficulty. If no single change is enough, it reports the changes that remove the most answers, so the model can get there in two or three steps. When a room is unique but has **too few silent chests**, it reports statements that can be removed while keeping it unique. This came from a level 5 trace: the model could make a room unique with every chest speaking, but each time it silenced two chests the room became ambiguous again, and it could not find the statement that settled it. Searching for that statement is easy for code and hard for a model.

It also reports **design problems**: two chests making the same statement, or a statement the mimic count settles on its own (for example "The mimics fill a diagonal line" when the room has 1 mimic, which is always false, so the speaker is obviously a Mimic). Neither makes a room wrong, but both make it worse, so each comes with replacement suggestions, and suggestions never include them.

The suggestions came from error analysis. On 3x3 rooms, `gpt-4.1-mini` was poor at working out whether its own compound statements ("exactly one of...", "the same kind", patterns) were true. Told that a statement was wrong, it moved the mimic and rewrote *every* statement, which broke different ones, and it never converged. With replacements in hand, it can fix one statement at a time.

If the room has exactly one answer but **not the one the model intended**, it is still a valid room: the solver's answer becomes the room's answer, and the report tells the model it may submit it as it is. (In testing, `gpt-4.1-mini` treated this case as a failure and went in circles, switching its intended answer back and forth between two layouts that were each already valid.)

Like the tutorial's tools, every expected problem is **returned as a message**, never raised, so the model can read it and correct itself.

The model gives the mimics as a list (`"mimics": ["B", "D"]`); every other chest is Normal. That is simpler for the model than a full answer dictionary.

In [10]:
def _normalize(grid_rows, grid_cols, chests, mimics, statements, show_mimic_count):
    # Validate a draft and convert it to the shared room format. Returns (room, errors).
    errors = []
    brief = SESSION["brief"]
    ids = [c["id"] for c in chests]
    if len(set(ids)) != len(ids):
        errors.append("Chest ids must be unique.")
    cells = [(c["row"], c["col"]) for c in chests]
    if len(set(cells)) != len(cells):
        errors.append("Two chests share a grid cell.")
    for c in chests:
        if not (0 <= c["row"] < grid_rows and 0 <= c["col"] < grid_cols):
            errors.append(f"Chest {c['id']} at row {c['row']}, col {c['col']} is outside the "
                          f"{grid_rows}x{grid_cols} grid (rows and cols start at 0).")
    if not mimics:
        errors.append("A room needs at least one mimic.")
    for m in mimics:
        if m not in ids:
            errors.append(f"Mimic {m} is not one of the chests.")
    speakers = [s["speaker"] for s in statements]
    if len(set(speakers)) != len(speakers):
        errors.append("Each chest can make at most one statement.")
    for s in statements:
        t, tg, sp = s["type"], s["targets"], s["speaker"]
        if sp not in ids:
            errors.append(f"Unknown speaker {sp}.")
        if t not in brief["allowed_types"]:
            errors.append(f"Statement type {t} isn't allowed in this brief "
                          f"(allowed: {brief['allowed_types']}).")
        if t == "pattern":
            if len(tg) != 1 or tg[0] not in PATTERNS:
                errors.append(f"{sp}: a pattern statement's target must be one of {list(PATTERNS)}.")
            continue
        if len(tg) != ARITY[t]:
            errors.append(f"{sp}: {t} takes {ARITY[t]} target(s), got {len(tg)}.")
        if any(x not in ids for x in tg):
            errors.append(f"{sp}'s statement names a chest that doesn't exist.")
        if sp in tg:
            errors.append(f"{sp} talks about itself, which isn't allowed.")
        if len(set(tg)) != len(tg):
            errors.append(f"{sp} names the same chest twice.")
    if errors:
        return None, errors

    # Single-target statements about a neighbour get direction words ("the chest to my right")
    pos = {c["id"]: (c["row"], c["col"]) for c in chests}
    at = {p: i for i, p in pos.items()}
    stmts = []
    for s in statements:
        s = {"speaker": s["speaker"], "type": s["type"], "targets": list(s["targets"])}
        if ARITY[s["type"]] == 1:
            r, c = pos[s["speaker"]]
            for (dr, dc), word in DIRECTIONS.items():
                if at.get((r + dr, c + dc)) == s["targets"][0]:
                    s["phrase"] = word
        stmts.append(s)
    order = {c: i for i, c in enumerate(ids)}
    stmts.sort(key=lambda s: order[s["speaker"]])
    solution = {c: ("mimic" if c in mimics else "normal") for c in ids}
    room = {"id": None, "grid": {"rows": grid_rows, "cols": grid_cols},
            "chests": [dict(c) for c in chests], "statements": stmts,
            "hints": {"mimic_count": len(set(mimics)), "show_mimic_count": show_mimic_count},
            "solution": solution, "rules": RULES, "instructions": INSTRUCTIONS}
    return room, []


def _fitting_statements(room, speaker, prefer_type=None, n=8):
    # Statements this speaker could make that fit the room's intended answer
    # (true for a Normal chest, false for a Mimic). prefer_type options come first.
    # Never suggests a statement another chest already makes, or a giveaway.
    pos = {c["id"]: (c["row"], c["col"]) for c in room["chests"]}
    grid = grid_of(room)
    intended = {c: v == "mimic" for c, v in room["solution"].items()}
    others = [c for c in pos if c != speaker]
    taken = {statement_key(st) for st in room["statements"] if st["speaker"] != speaker}
    giveaway = SESSION.setdefault("giveaway_cache", {})
    options = []
    for t in SESSION["brief"]["allowed_types"]:
        if t == "pattern":
            candidates = [[p] for p in PATTERNS]
        elif ARITY[t] == 1:
            candidates = [[o] for o in others]
        else:
            candidates = [[x, y] for x in others for y in others if x < y or t == "if_then" and x != y]
        for tg in candidates:
            s = {"speaker": speaker, "type": t, "targets": tg}
            if statement_truth(s, intended, pos, grid) == intended[speaker]:
                continue
            key = statement_key(s)
            if key in taken:
                continue
            cache_key = (key, room["hints"]["mimic_count"], room["hints"]["show_mimic_count"],
                         tuple(sorted(pos.items())))
            if cache_key not in giveaway:
                giveaway[cache_key] = is_giveaway(s, room)
            if not giveaway[cache_key]:
                options.append({"type": t, "targets": tg, "reads": statement_text(s)})
    SESSION["rng"].shuffle(options)
    options.sort(key=lambda o: o["type"] != prefer_type)   # stable: preferred type first
    return options[:n]


def _suggest_changes(room, n=3):
    # Search one-statement changes that make the room uniquely solvable with its intended
    # answer, without breaking the brief's silent-chest count. A language model can't run
    # this search in its head; the harness can, in well under a second.
    brief = SESSION["brief"]
    want_silent = brief["silent_chests"]
    lo, hi = brief["target_difficulty"]
    speakers = [st["speaker"] for st in room["statements"]]
    silent_ids = [c["id"] for c in room["chests"] if c["id"] not in speakers]
    silent_now = len(silent_ids)
    trials = []
    if silent_now == want_silent:          # replace one existing statement
        for sp in speakers:
            for o in _fitting_statements(room, sp, n=200):
                new = {"speaker": sp, "type": o["type"], "targets": o["targets"]}
                stmts = [new if st["speaker"] == sp else st for st in room["statements"]]
                trials.append(("replace", sp, new, stmts))
    elif silent_now > want_silent:         # give a silent chest a statement
        for sp in silent_ids:
            for o in _fitting_statements(room, sp, n=200):
                new = {"speaker": sp, "type": o["type"], "targets": o["targets"]}
                trials.append(("add", sp, new, room["statements"] + [new]))
    else:                                  # too few silent chests: remove a statement
        for sp in speakers:
            stmts = [st for st in room["statements"] if st["speaker"] != sp]
            trials.append(("remove", sp, None, stmts))
    SESSION["rng"].shuffle(trials)
    now = len(solve_all(room))
    found, partial = [], []
    for action, sp, new, stmts in trials:
        trial = dict(room, statements=stmts)
        sols = solve_all(trial)
        if room["solution"] not in sols:
            continue
        if len(sols) == 1:
            d = GeneratorAgent._difficulty(trial, solve_with_trace(trial))
            found.append((0 if lo <= d <= hi else min(abs(d - lo), abs(d - hi)), action, sp, new, d))
            if len(found) >= 25:
                break
        elif len(sols) < now:
            partial.append((len(sols), action, sp, new))
    if found:
        found.sort(key=lambda f: f[0])
        picks = [(a, sp, new, {"result": "unique", "difficulty_after": d})
                 for _, a, sp, new, d in found[:n]]
    else:
        # No single change is enough: suggest the ones that remove the most answers.
        partial.sort(key=lambda f: f[0])
        picks = [(a, sp, new, {"result": f"{m} answers left (now {now})"})
                 for m, a, sp, new in partial[:n]]
    out = []
    for action, sp, new, extra in picks:
        item = {"action": action, "speaker": sp, **extra}
        if new:
            item["statement"] = {"type": new["type"], "targets": new["targets"]}
        out.append(item)
    return out


def check_room(grid_rows: int, grid_cols: int, chests: list, mimics: list,
               statements: list, show_mimic_count: bool) -> dict:
    SESSION["checks"] += 1
    room, errors = _normalize(grid_rows, grid_cols, chests, mimics, statements, show_mimic_count)
    if errors:
        SESSION["draft"], SESSION["draft_unique"] = None, False
        return {"status": "format_error", "problems": errors}

    pos = {c["id"]: (c["row"], c["col"]) for c in room["chests"]}
    intended = {c: v == "mimic" for c, v in room["solution"].items()}
    report = {"status": None, "problems": []}

    # 1. Does every statement fit the intended answer? For each one that doesn't,
    #    suggest replacements that do, so the model can swap one in instead of guessing.
    fixes = {}
    for s in room["statements"]:
        true = statement_truth(s, intended, pos, grid_of(room))
        if true == intended[s["speaker"]]:
            role = "a Mimic, so its statement must be FALSE" if intended[s["speaker"]] \
                   else "Normal, so its statement must be TRUE"
            report["problems"].append(
                f"{s['speaker']} is {role}, but \"{statement_text(s)}\" is "
                f"{'true' if true else 'false'} in your intended answer.")
            fixes[s["speaker"]] = [{"type": o["type"], "targets": o["targets"]}
                                   for o in _fitting_statements(room, s["speaker"], s["type"], n=3)]
    if fixes:
        report["suggested_fixes"] = fixes
        report["fix_advice"] = ("Keep the same mimics. Replace only the statements listed in "
                                "suggested_fixes, using one of the suggestions (each one fits your "
                                "intended answer), and check again.")

    # 1b. Duplicate statements and giveaways (statements the mimic count settles alone)
    design = design_problems(room)
    if design:
        report["design_problems"] = design
        seen = set()
        for st in room["statements"]:
            key = statement_key(st)
            if (key in seen or is_giveaway(st, room)) and st["speaker"] not in fixes:
                fixes[st["speaker"]] = [{"type": o["type"], "targets": o["targets"]}
                                        for o in _fitting_statements(room, st["speaker"], st["type"], n=3)]
            seen.add(key)
        report["suggested_fixes"] = fixes
        report["fix_advice"] = ("Keep the same mimics. Replace the statements listed in "
                                "suggested_fixes with one of the suggestions, and check again.")

    # 2. Does the draft follow the brief? (grid, chests, mimics, silent chests, count hint)
    brief = SESSION["brief"]
    rows, cols = map(int, brief["grid"].split("x"))
    silent = len(room["chests"]) - len(room["statements"])
    brief_problems = []
    if (grid_rows, grid_cols) != (rows, cols):
        brief_problems.append(f"The grid is {grid_rows}x{grid_cols}; the brief asks for {brief['grid']}.")
    if len(room["chests"]) != brief["num_chests"]:
        brief_problems.append(f"There are {len(room['chests'])} chests; the brief asks for {brief['num_chests']}.")
    if room["hints"]["mimic_count"] != brief["num_mimics"]:
        brief_problems.append(f"There are {room['hints']['mimic_count']} mimics; the brief asks for {brief['num_mimics']}.")
    if silent != brief["silent_chests"]:
        brief_problems.append(f"{silent} chest(s) say nothing; the brief asks for exactly "
                              f"{brief['silent_chests']} silent chest(s).")
    if show_mimic_count != brief["show_mimic_count"]:
        brief_problems.append(f"show_mimic_count should be {brief['show_mimic_count']}.")
    report["brief_problems"] = brief_problems

    # 3. How many answers does the room have?
    sols = solve_all(room, limit=3)
    SESSION["draft"] = room
    SESSION["draft_unique"] = len(sols) == 1
    if len(sols) == 1 and sols[0] == room["solution"]:
        report["status"] = "unique"
        report["message"] = "Exactly one answer, and it is your intended answer."
    elif len(sols) == 1:
        # Still a valid room: the solver's answer becomes the room's answer.
        # (The intended answer is only a planning aid; chasing it made the model go in circles.)
        room["solution"] = sols[0]
        room["hints"]["mimic_count"] = sum(v == "mimic" for v in sols[0].values())
        report["problems"] = []
        report["brief_problems"] = [p for p in brief_problems if "mimics;" not in p]
        if room["hints"]["mimic_count"] != brief["num_mimics"]:
            report["brief_problems"].append(
                f"The room's answer has {room['hints']['mimic_count']} mimics; "
                f"the brief asks for {brief['num_mimics']}.")
        brief_problems = report["brief_problems"]
        report["status"] = "unique"
        report["message"] = ("Exactly one answer, but not the one you intended. The room is still "
                             "valid, and its answer is the one below. Submit it as it is (if it has "
                             "no brief_problems) rather than rewriting it.")
        report["answer"] = sols[0]
    elif not sols:
        report["status"] = "contradictory"
        report["message"] = "No answer fits all the clues."
    else:
        a, b = sols[0], sols[1]
        report["status"] = "ambiguous"
        report["message"] = (f"{'3 or more' if len(sols) > 2 else 2} answers fit. Add or change "
                             "a statement that tells the differing chests apart.")
        report["competing_answers"] = [a, b]
        report["differing_chests"] = [c for c in a if a[c] != b[c]]
        if not report["problems"]:
            changes = _suggest_changes(room)
            if changes:
                report["suggested_changes"] = changes
                report["change_advice"] = ("Each suggested change was tested (see its result). "
                                           "Apply ONE and check again; if the result says answers "
                                           "are left, check_room will suggest the next step.")

    # 4. Difficulty against the brief
    if sols:
        trace = solve_with_trace(room)
        lo, hi = SESSION["brief"]["target_difficulty"]
        report["target_difficulty"] = [lo, hi]
        if SESSION["draft_unique"]:
            d = GeneratorAgent._difficulty(room, trace)
            report["difficulty"] = d
            if d < lo:
                report["advice"] = "Too easy: use more indirect clues or fewer direct ones."
            elif d > hi:
                report["advice"] = "Too hard: add a clue that gives the player a foothold."
            else:
                report["advice"] = "Difficulty fits the brief."
            if brief_problems or design:
                report["advice"] += " Fix the brief_problems and design_problems before submitting."
            elif lo <= d <= hi:
                report["advice"] += " Ready to submit."
            # Unique, but too few silent chests: find statements that can go.
            if silent < brief["silent_chests"]:
                changes = _suggest_changes(room)
                if changes:
                    report["suggested_changes"] = changes
                    report["change_advice"] = ("The room needs more silent chests. Each suggested "
                                               "removal was tested: the room stays uniquely "
                                               "solvable without that statement. Apply one and "
                                               "check again.")
    report["room_as_player_sees_it"] = [
        f"{s['speaker']}: \"{statement_text(s)}\"" for s in room["statements"]]
    return report

### Tool 3: `statement_options`

A helper for when the model is stuck: it lists statements that a chest could make under the last checked draft's intended answer (true ones for a Normal chest, false ones for a Mimic). It returns a **random sample**, so it suggests without dictating.

In [11]:
def statement_options(speaker: str, statement_type: str | None = None) -> dict:
    room = SESSION["draft"]
    if room is None:
        return {"error": "Check a valid draft first, so I know the layout and intended answer."}
    if speaker not in {c["id"] for c in room["chests"]}:
        return {"error": f"No chest {speaker} in the draft."}
    options = _fitting_statements(room, speaker, statement_type, n=8)
    if statement_type:
        options = [o for o in options if o["type"] == statement_type]
    role = "a Mimic (its statement must be false)" if room["solution"][speaker] == "mimic" \
           else "Normal (its statement must be true)"
    return {"speaker": speaker, "role": role, "options": options}

### Tool 4: `submit_room`

The **guardrail that matters most**: the room is accepted only if the last checked draft has exactly one answer. The model cannot talk its way past the solver.

In [12]:
def submit_room(design_note: str) -> dict:
    if SESSION["draft"] is None:
        return {"accepted": False, "message": "No valid draft has been checked. Use check_room first."}
    if not SESSION["draft_unique"]:
        return {"accepted": False, "message": "The last checked draft does not have exactly one "
                "answer. Revise it and check again."}
    SESSION["final"] = dict(SESSION["draft"], design_note=design_note)
    return {"accepted": True, "message": "The room has been delivered to the player."}

### Test the tools directly

Always test tools on their own before giving them to an agent, including the cases that should fail. We play the agent's part by hand: a format error, an ambiguous room, a fixed room, and a submit.

In [13]:
brief = start_session(2, seed=1)
print("Brief:", brief, "\n")

chests = [{"id": "A", "row": 0, "col": 0}, {"id": "B", "row": 0, "col": 1},
          {"id": "C", "row": 1, "col": 0}, {"id": "D", "row": 1, "col": 1}]

# 1. Format error: a chest outside the grid, and a chest talking about itself
bad = [{"id": "A", "row": 0, "col": 0}, {"id": "B", "row": 5, "col": 1}]
print(check_room(2, 2, bad, ["B"], [{"speaker": "A", "type": "is_mimic", "targets": ["A"]}], True), "\n")

# 2. Ambiguous: only two clues
two_clues = [{"speaker": "A", "type": "not_mimic", "targets": ["B"]},
             {"speaker": "D", "type": "is_mimic", "targets": ["C"]}]
print(check_room(2, 2, chests, ["C"], two_clues, True), "\n")

# 3. Submitting an ambiguous room is rejected
print(submit_room("test"), "\n")

# 4. Fixed: the room from the pitch deck
pitch = [{"speaker": "A", "type": "not_mimic", "targets": ["B"]},
         {"speaker": "B", "type": "not_mimic", "targets": ["D"]},
         {"speaker": "C", "type": "is_mimic", "targets": ["D"]},
         {"speaker": "D", "type": "is_mimic", "targets": ["C"]}]
print(check_room(2, 2, chests, ["C"], pitch, True), "\n")
print(statement_options("A"), "\n")
print(submit_room("C and D accuse each other; the count hint settles it."))

Brief: {'level': 2, 'grid': '2x2', 'num_chests': 4, 'num_mimics': 2, 'allowed_types': ['is_mimic', 'not_mimic', 'either_one', 'same_kind'], 'show_mimic_count': True, 'silent_chests': 0, 'target_difficulty': [2, 3], 'twist': "Let one chest's statement be true but useless on its own."} 

{'status': 'format_error', 'problems': ['Chest B at row 5, col 1 is outside the 2x2 grid (rows and cols start at 0).', "A talks about itself, which isn't allowed."]} 

{'status': 'ambiguous', 'problems': [], 'brief_problems': ['There are 1 mimics; the brief asks for 2.', '2 chest(s) say nothing; the brief asks for exactly 0 silent chest(s).'], 'message': '2 answers fit. Add or change a statement that tells the differing chests apart.', 'competing_answers': [{'A': 'normal', 'B': 'normal', 'C': 'normal', 'D': 'mimic'}, {'A': 'normal', 'B': 'normal', 'C': 'mimic', 'D': 'normal'}], 'differing_chests': ['C', 'D'], 'suggested_changes': [{'action': 'add', 'speaker': 'C', 'result': 'unique', 'difficulty_after': 

### Pedagogical note: who checks the model?

The model never decides whether a room is valid. It **proposes**; `check_room` **measures**; `submit_room` **enforces**. That split is what lets us raise the temperature for creativity without risking a broken puzzle reaching a player.

# 4. Build the Agent with LangChain

## Step 1: Turn the functions into tools

As in the tutorial, `@tool` turns each function into a tool: the name, the type hints (a `Literal` becomes a list of allowed values), and the docstring become what the model reads.

`check_room` takes structured arguments, so we describe a chest and a statement with small **Pydantic** models. LangChain turns them into a JSON schema, and validates the model's arguments before our function runs.

In [14]:
from pydantic import BaseModel, Field
from langchain.tools import tool

StatementType = Literal["is_mimic", "not_mimic", "either_one", "if_then", "same_kind", "pattern"]

class Chest(BaseModel):
    id: str = Field(description="Chest id: A, B, C... in reading order")
    row: int = Field(description="Row, starting at 0 (top)")
    col: int = Field(description="Column, starting at 0 (left)")

class Statement(BaseModel):
    speaker: str = Field(description="Id of the chest making the statement")
    type: StatementType
    targets: list[str] = Field(description="Chest ids the statement is about; "
                                           "for 'pattern', one pattern name")

@tool("get_design_brief")
def design_brief_tool() -> dict:
    '''Get the design brief for this room: grid, chest count, mimic count, allowed statement
    types, silent chests, mimic-count setting, target difficulty, and twist. Call this first.'''
    return get_design_brief()

@tool("check_room")
def check_room_tool(grid_rows: int, grid_cols: int, chests: list[Chest], mimics: list[str],
                    statements: list[Statement], show_mimic_count: bool) -> dict:
    '''Check a complete draft room. mimics lists the chests you intend to be mimics; every
    other chest is Normal. Returns format problems, statements that break your intended
    answer, brief_problems (where the draft doesn't match the brief), whether the room has
    exactly one answer (and if not, the competing answers and the chests they disagree on),
    and the difficulty compared with the brief.
    Use it after every draft and every revision.'''
    return check_room(grid_rows, grid_cols, [c.model_dump() for c in chests], mimics,
                      [s.model_dump() for s in statements], show_mimic_count)

@tool("statement_options")
def statement_options_tool(speaker: str, statement_type: StatementType | None = None) -> dict:
    '''List some valid statements a chest could make under the last checked draft's intended
    answer. Use it when you are stuck on a revision. Returns a random sample.'''
    return statement_options(speaker, statement_type)

@tool("submit_room")
def submit_room_tool(design_note: str) -> dict:
    '''Deliver the last checked draft to the player. Only accepted if check_room reported
    exactly one answer. design_note: one sentence describing the key deduction.'''
    return submit_room(design_note)

TOOLS = [design_brief_tool, check_room_tool, statement_options_tool, submit_room_tool]
print([t.name for t in TOOLS])

['get_design_brief', 'check_room', 'statement_options', 'submit_room']


## Step 2: The model

`ChatOpenAI` pointed at OpenRouter, the same as the tutorial. With `USE_OFFLINE_MODEL = True`, the scripted stand-in takes its place; nothing else in the agent changes.

In [15]:
if USE_OFFLINE_MODEL:
    from offline_model import OfflineStandInModel
    model = OfflineStandInModel()
else:
    from langchain_openai import ChatOpenAI
    model = ChatOpenAI(
        model=MODEL,
        temperature=TEMPERATURE,
        base_url=OPENROUTER_BASE_URL,
        api_key=os.getenv("OPENROUTER_API_KEY"),
    )

## Step 3: The system prompt is the agent's policy

The policy states what the agent is for, the game's rules (which no tool provides), the room format, and how to work. As in the tutorial, we ask for **one short sentence before each tool call**, which gives us the **Thought** part of the trace.

In [16]:
pattern_help = "\n".join(f"    {name}: \"{desc}\"" for name, (_, desc) in PATTERNS.items())
pattern_rules = "\n".join(f"  - {r}" for r in PATTERN_RULES)

SYSTEM_PROMPT = f'''You are the Puzzle Generator for Mimic Logic, a dungeon logic game. You design
one room at a time, following a design brief.

Game rules:
- Chests sit on a grid. Each chest is Normal (always tells the truth) or a Mimic (always lies).
- Each chest makes at most one statement; some chests stay silent.
- A room must have exactly one answer.

Statement types (type, targets):
- is_mimic [X]: "Chest X is a mimic."         - not_mimic [X]: "Chest X is not a mimic."
- either_one [X, Y]: "Exactly one of Chest X and Chest Y is a mimic."
- if_then [X, Y]: "If Chest X is a mimic, then Chest Y is a mimic."
- same_kind [X, Y]: "Chest X and Chest Y are the same kind."
- pattern [name], where name is one of:
{pattern_help}
  What the patterns mean exactly:
{pattern_rules}
A chest never talks about itself. No two chests may make the same statement, and avoid
statements the mimic count settles on its own (e.g. a row, column or diagonal pattern when
there is only 1 mimic): they give the answer away. check_room flags both as design_problems.
Statements about a neighbouring chest are worded with
directions automatically ("the chest to my right").

How to work:
- Before each tool call, write one short sentence saying what you are about to do and why.
- First call get_design_brief. Follow the brief: grid, number of chests, number of mimics,
  allowed statement types, silent chests, mimic-count setting, and the twist.
- Decide the hidden answer and the key deduction FIRST, then write statements that lead to it.
  A Normal chest's statement must be TRUE; a Mimic's statement must be FALSE.
- Use ids A, B, C... in reading order (left to right, top to bottom). Rows and cols start at 0.
- Call check_room with the whole draft. Fix every problem it reports. If it reports a unique
  room whose answer differs from the one you intended, that room is still valid: submit it
  (if there are no brief_problems) instead of rewriting it. If it is ambiguous,
  change or add a statement about the chests the competing answers disagree on.
- While revising, keep the same mimics and change only the statements check_room lists as
  broken. Use suggested_fixes: each suggestion fits your intended answer. Do not move the
  mimics or rewrite statements that are already fine.
- If check_room returns suggested_changes, apply one of them: each was tested, and its
  result says whether it makes the room uniquely solvable or removes some of the answers.
- Use statement_options only if you are stuck.
- When check_room reports a unique room with no brief_problems and a difficulty that fits
  the brief, call submit_room.
- Be inventive: vary the layout and the kinds of claims; avoid rooms where every chest says
  the same kind of thing.
- Every reply must call a tool until submit_room is accepted. Never stop to summarise the
  brief or describe a plan in plain text: put your reasoning in the one short sentence before
  the tool call, and keep going.
- After submit_room is accepted, reply with one sentence and stop.'''

print(SYSTEM_PROMPT)

You are the Puzzle Generator for Mimic Logic, a dungeon logic game. You design
one room at a time, following a design brief.

Game rules:
- Chests sit on a grid. Each chest is Normal (always tells the truth) or a Mimic (always lies).
- Each chest makes at most one statement; some chests stay silent.
- A room must have exactly one answer.

Statement types (type, targets):
- is_mimic [X]: "Chest X is a mimic."         - not_mimic [X]: "Chest X is not a mimic."
- either_one [X, Y]: "Exactly one of Chest X and Chest Y is a mimic."
- if_then [X, Y]: "If Chest X is a mimic, then Chest Y is a mimic."
- same_kind [X, Y]: "Chest X and Chest Y are the same kind."
- pattern [name], where name is one of:
    same_row: "All the mimics are in the same row."
    same_col: "All the mimics are in the same column."
    l_shape: "The mimics form an L shape."
    diagonal: "The mimics fill a diagonal line."
    none_adjacent: "No two mimics are next to each other."
  What the patterns mean exactly:
  - Sa

## Step 4: Guardrails as middleware

The tutorial's three guardrails, with budgets sized for this task, plus one of our own.

**`keep_going_until_submitted`.** In `create_agent`, a model reply with **no tool call** counts as the final answer and ends the run. In testing, `gpt-4.1-mini` often read the brief and then replied with a plain-text summary of it, which ended the run before any room was drafted. This custom `after_model` hook checks every model reply: if it has no tool call and no room has been accepted yet, the harness adds a reminder and sends the model back (`jump_to="model"`). `ModelCallLimitMiddleware` still caps the total, so this cannot loop forever. The hook only looks at the model's own replies: other middleware, such as the tool-call limit, can add tool messages after a reply, and those turns are left alone.

**`one_tool_call_at_a_time`.** OpenAI models may make several tool calls in one turn. In testing, the model checked two different drafts at once, and `submit_room` then delivered whichever happened to be checked last: the worse of the two. This `wrap_model_call` middleware turns parallel tool calls off, so every draft is checked, read, and then either revised or submitted.

The other three are the same as the tutorial. In testing, `gpt-4.1-mini` needed 5 to 12 tool calls per room (it often breaks a statement while revising and has to fix it). The limits leave room for that while stopping a confused agent from looping, and if it runs out, the harness falls back to the algorithmic generator (Step 5), so the player still gets a room.

In [17]:
from langchain.agents import create_agent
from langchain.agents.middleware import (
    ModelCallLimitMiddleware,
    ToolCallLimitMiddleware,
    ToolRetryMiddleware,
)

from langchain.agents.middleware import after_model, wrap_model_call
from langchain_core.messages import HumanMessage

@wrap_model_call
def one_tool_call_at_a_time(request, handler):
    # submit_room delivers "the last checked draft". If the model checks two drafts in
    # one turn, which one is "last" is an accident, so turn parallel tool calls off.
    settings = {**(request.model_settings or {}), "parallel_tool_calls": False}
    return handler(request.override(model_settings=settings))

@after_model(can_jump_to=["model"])
def keep_going_until_submitted(state, runtime):
    # The model may not end the run before a room is accepted: if it replies with
    # plain text and no tool call, send it back with a reminder.
    # Only act on a plain model reply. Other middleware (e.g. the tool-call limit) can add
    # tool messages after the model's reply; leave those turns alone.
    last = state["messages"][-1]
    if last.type != "ai":
        return None
    if not last.tool_calls and SESSION.get("final") is None:
        return {"messages": [HumanMessage(
                    "No room has been submitted yet, so you are not done. Continue: draft the "
                    "room, call check_room, and call submit_room when it has exactly one answer.")],
                "jump_to": "model"}
    return None

GUARDRAILS = [
    one_tool_call_at_a_time,
    keep_going_until_submitted,
    ModelCallLimitMiddleware(run_limit=24),
    ToolCallLimitMiddleware(run_limit=20),
    ToolRetryMiddleware(max_retries=2, initial_delay=0.5),
]

agent = create_agent(
    model=model,
    tools=TOOLS,
    system_prompt=SYSTEM_PROMPT,
    middleware=GUARDRAILS,
)

## Step 5: A readable trace and `generate_room`

`build_trace` and `print_trace` are the tutorial's helpers, with one change: `check_room` calls carry a whole room, so the trace prints a short summary of each draft instead of the full JSON.

`generate_room` is the function the game will call. It starts a session, runs the agent, and returns the room plus its trace. If the agent fails to deliver (a guardrail stopped it, or the API failed), it **falls back** to the algorithmic generator and records that in the room's `meta`.

In [18]:
def build_trace(result):
    # Turn the agent's message history into a list of ReAct steps
    steps = []
    seen_question = False
    for message in result["messages"]:
        if message.type == "human":
            # the first human message is the request; later ones are harness reminders
            steps.append({"step": "harness" if seen_question else "question", "text": message.text})
            seen_question = True
        elif message.type == "ai" and message.tool_calls:
            if message.text:
                steps.append({"step": "thought", "text": message.text})
            for call in message.tool_calls:
                steps.append({"step": "action", "tool": call["name"], "args": call["args"]})
        elif message.type == "ai":
            steps.append({"step": "final", "text": message.text})
        elif message.type == "tool":
            steps.append({"step": "observation", "tool": message.name, "text": message.text})
    # a text-only reply that the harness sent back is a thought, not the final answer
    for s in steps[:-1]:
        if s["step"] == "final":
            s["step"] = "thought"
    return steps

def summarize_args(tool_name, args):
    if tool_name == "check_room":
        return (f"check_room(grid {args.get('grid_rows')}x{args.get('grid_cols')}, "
                f"{len(args.get('chests', []))} chests, mimics {args.get('mimics')}, "
                f"{len(args.get('statements', []))} statements)")
    return f"{tool_name}({args})"

def compact_json(text, limit=400):
    try:
        text = json.dumps(json.loads(text))
    except (ValueError, TypeError):
        pass
    return text if len(text) <= limit else text[:limit] + " ..."

def print_trace(steps):
    labels = {"question": "Question", "thought": "Thought", "action": "Action",
              "observation": "Observation", "final": "Final answer", "harness": "Harness"}
    for s in steps:
        if s["step"] == "action":
            detail = summarize_args(s["tool"], s["args"])
        elif s["step"] == "observation":
            detail = compact_json(s["text"])
        else:
            detail = s["text"]
        print(f"{labels[s['step']] + ':':<14}{detail}")


def generate_room(agent, level=2, seed=None, show_trace=True):
    # Run the agent once and return (room, steps). Falls back to the algorithmic generator.
    brief = start_session(level, seed)
    request = "Design the next room."
    error = None
    try:
        result = agent.invoke({"messages": [{"role": "user", "content": request}]})
        steps = build_trace(result)
    except Exception as exc:
        steps = [{"step": "question", "text": request}]
        error = f"{type(exc).__name__}: {exc}"
    if show_trace:
        print_trace(steps)

    room = SESSION["final"]
    source = "agent"
    if room is None:
        source = "fallback"
        room = GeneratorAgent(seed=seed).generate(
            profile={"level": brief["level"]})
        if show_trace:
            print(f"\n[The agent did not deliver a room{': ' + error if error else ''}. "
                  "Using the algorithmic generator.]")

    room = dict(room, id=make_room_id(brief["level"]), rules_version=RULES_VERSION)
    trace = solve_with_trace(room)
    room["meta"] = {
        "source": source,
        "brief": brief,
        "design_note": room.pop("design_note", None),
        "skills": sorted({s["type"] for s in room["statements"]}),
        "difficulty": GeneratorAgent._difficulty(room, trace),
        "assumption_depth": trace["max_depth"],
        "tool_calls": sum(s["step"] == "action" for s in steps),
        "reminders": sum(s["step"] == "harness" for s in steps),
        "checks": SESSION["checks"],
        "error": error,
    }
    return room, steps

## Where is the harness?

| Harness piece | Here |
|---|---|
| The loop | `create_agent` |
| Tool schemas | `@tool` + Pydantic models |
| Budgets | `ModelCallLimitMiddleware`, `ToolCallLimitMiddleware` |
| No stopping early | `keep_going_until_submitted` (custom `after_model` hook) |
| One draft at a time | `one_tool_call_at_a_time` (custom `wrap_model_call` middleware) |
| Tool failures | `ToolRetryMiddleware`; expected problems returned as messages |
| Output guardrail | `submit_room` accepts only rooms the solver verified |
| Graceful failure | `generate_room` falls back to the algorithmic generator |
| A trace you can keep | `build_trace` returns steps as data |

# 5. Generate Rooms

A level 2 room. Read the trace: each **Thought** is the model's reasoning, each **Observation** is what the solver reported back.

In [19]:
room, steps = generate_room(agent, level=2)
print()
print(render(room))
print("\nMeta:", {k: v for k, v in room["meta"].items() if k != "brief"})

Question:     Design the next room.
Thought:      I will start by retrieving the design brief to know the grid size, number of chests, number of mimics, allowed statement types, and other constraints.
Action:       get_design_brief({})
Observation:  {"level": 2, "grid": "2x2", "num_chests": 4, "num_mimics": 1, "allowed_types": ["is_mimic", "not_mimic", "either_one", "same_kind"], "show_mimic_count": true, "silent_chests": 0, "target_difficulty": [2, 3], "twist": "Two chests of the same kind should be far apart on the grid."}
Thought:      I will choose one mimic and design a unique answer with one mimic in a 2x2 grid, then draft statements fitting allowed types and the twist of two chests of the same kind far apart.
Action:       check_room(grid 2x2, 4 chests, mimics ['B'], 3 statements)
Observation:  {"status": "format_error", "problems": ["C: same_kind takes 2 target(s), got 1."]}
Thought:      The statement for chest C has the wrong number of targets; I will fix it to refer to both 

A level 4 room: a bigger grid, all six statement types, and a silent chest or two.

In [20]:
room, steps = generate_room(agent, level=4)
print()
print(render(room))
print("\nTwist:", room["meta"]["brief"]["twist"])
print("Design note:", room["meta"]["design_note"])

Question:     Design the next room.
Thought:      I will get the design brief to know the grid size, chest and mimic counts, allowed statement types, and other constraints for this room.
Action:       get_design_brief({})
Observation:  {"level": 4, "grid": "3x3", "num_chests": 6, "num_mimics": 1, "allowed_types": ["is_mimic", "not_mimic", "either_one", "if_then", "same_kind", "pattern"], "show_mimic_count": true, "silent_chests": 0, "target_difficulty": [3, 5], "twist": "A Normal chest should look the most suspicious at first glance."}
Thought:      I will choose one mimic and five normal chests on a 3x3 grid and decide a unique answer where the mimic is the most suspicious chest, then I will draft statements accordingly.
Action:       check_room(grid 3x3, 6 chests, mimics ['E'], 5 statements)
Observation:  {"status": "unique", "problems": [], "brief_problems": ["1 chest(s) say nothing; the brief asks for exactly 0 silent chest(s)."], "message": "Exactly one answer, and it is your inte

In [21]:
# Show the answer and the solver's step-by-step reasoning (what the demo shows side by side)
print("Answer:", room["solution"])
print("\n".join(solve_with_trace(room)["trace"]))

Answer: {'A': 'normal', 'B': 'normal', 'C': 'normal', 'D': 'normal', 'E': 'mimic', 'F': 'normal'}
Assume A is a Mimic:
  From the mimic-count hint: B, C, D, E, F are Normal.
  Contradiction.
Assume A is Normal:
  Assume D is a Mimic:
    From the mimic-count hint: B, C, E, F are Normal.
    Contradiction.
  Assume D is Normal:
    From A's statement: E is a Mimic.
    From B's statement: B is Normal.
    From C's statement: C is Normal.
    From F's statement: F is Normal.
    Everything fits.


# 6. Evaluate the Agent

A generator has no single correct answer to compare against, but most of what makes a room *good* can still be checked **objectively and for free**:

| Check | Question |
|---|---|
| **delivered** | Did the agent deliver a room itself, without the fallback? |
| **valid** | Re-solved independently by brute force: exactly one answer, equal to the stored solution? |
| **brief** | Grid size, chest count, mimic count, silent chests, mimic-count setting, and only allowed statement types? |
| **design** | No two chests make the same statement, and no statement is settled by the mimic count alone? |
| **difficulty** | Is the difficulty inside the brief's target range? |
| **budget** | At most 12 tool calls? |

The **twist** is a creative constraint with no exact test, so we judge it with an **LLM judge** in Step 3.

## Step 1: A small test set

Each case is a difficulty level plus a fixed **seed**, so every run of the evaluation gets the same briefs. (The model's temperature is 0.7, so its rooms will still differ between runs; run the evaluation more than once before trusting a difference.)

In [22]:
TEST_CASES = [
    {"id": "level1_a", "level": 1, "seed": 11},
    {"id": "level2_a", "level": 2, "seed": 12},
    {"id": "level2_b", "level": 2, "seed": 13},
    {"id": "level3_a", "level": 3, "seed": 14},
    {"id": "level4_a", "level": 4, "seed": 15},
    {"id": "level4_b", "level": 4, "seed": 16},
    {"id": "level5_a", "level": 5, "seed": 17},
]
MAX_TOOL_CALLS = 12
print(f"{len(TEST_CASES)} test cases")

7 test cases


## Step 2: Run every case and check it

In [23]:
def check_room_record(room, steps):
    meta, brief = room["meta"], room["meta"]["brief"]
    rows, cols = map(int, brief["grid"].split("x"))
    sols = solve_all(room)
    types = {s["type"] for s in room["statements"]}
    silent = len(room["chests"]) - len(room["statements"])
    lo, hi = brief["target_difficulty"]
    checks = {
        "delivered": meta["source"] == "agent",
        "valid": len(sols) == 1 and sols[0] == room["solution"],
        "brief": (room["grid"] == {"rows": rows, "cols": cols}
                  and len(room["chests"]) == brief["num_chests"]
                  and room["hints"]["mimic_count"] == brief["num_mimics"]
                  and room["hints"]["show_mimic_count"] == brief["show_mimic_count"]
                  and silent == brief["silent_chests"]
                  and types <= set(brief["allowed_types"])),
        "design": not design_problems(room),
        "difficulty": lo <= meta["difficulty"] <= hi,
        "budget": meta["tool_calls"] <= MAX_TOOL_CALLS,
    }
    notes = []
    if meta["source"] == "agent" and not checks["brief"]:
        notes.append(f"brief: grid {room['grid']['rows']}x{room['grid']['cols']} vs {brief['grid']}, "
                     f"chests {len(room['chests'])} vs {brief['num_chests']}, "
                     f"mimics {room['hints']['mimic_count']} vs {brief['num_mimics']}, "
                     f"silent {silent} vs {brief['silent_chests']}")
    if not checks["design"]:
        notes.append("design: " + design_problems(room)[0][:60])
    if not checks["difficulty"]:
        notes.append(f"difficulty {meta['difficulty']} vs {lo}-{hi}")
    if meta["error"]:
        notes.append(meta["error"][:60])
    return checks, "; ".join(notes)

def evaluate_agent(agent, cases):
    records = []
    for case in cases:
        room, steps = generate_room(agent, case["level"], seed=case["seed"], show_trace=False)
        checks, note = check_room_record(room, steps)
        records.append({"id": case["id"], "room": room, "steps": steps,
                        "checks": checks, "note": note})
        print(".", end="")
    print()
    return records

def show_results(records):
    names = list(records[0]["checks"])
    mark = lambda ok: "pass" if ok else "FAIL"
    print(f"{'case':<17}" + "".join(f"{n:<11}" for n in names) + f"{'calls':<7}{'nudges':<8}note")
    for r in records:
        print(f"{r['id']:<17}" + "".join(f"{mark(r['checks'][n]):<11}" for n in names)
              + f"{r['room']['meta']['tool_calls']:<7}{r['room']['meta']['reminders']:<8}{r['note']}")
    for n in names:
        print(f"{n:>11}: {sum(r['checks'][n] for r in records)}/{len(records)} passed")

In [24]:
baseline_records = evaluate_agent(agent, TEST_CASES)
show_results(baseline_records)

.......
case             delivered  valid      brief      design     difficulty budget     calls  nudges  note
level1_a         pass       pass       pass       pass       pass       pass       5      0       
level2_a         pass       pass       pass       pass       pass       pass       7      0       
level2_b         pass       pass       pass       pass       pass       pass       5      0       
level3_a         pass       pass       pass       pass       pass       pass       4      0       
level4_a         pass       pass       pass       pass       pass       pass       8      0       
level4_b         pass       pass       pass       pass       pass       pass       3      0       
level5_a         pass       pass       pass       pass       pass       pass       6      0       
  delivered: 7/7 passed
      valid: 7/7 passed
      brief: 7/7 passed
     design: 7/7 passed
 difficulty: 7/7 passed
     budget: 7/7 passed


Read the table row by row before looking at the totals. Note that **valid** should always pass, even for a poor agent: either the agent's room passed `submit_room`, or the fallback produced one. A failure in **delivered** with a pass in **valid** means the *harness* saved the run, not the model. That is worth knowing, and exactly the kind of thing the totals hide.

## Step 3: An LLM judge for the twist

Did the room actually follow its twist (e.g. *"No chest directly calls another chest a mimic"*)? Some twists could be checked with code, but most need judgment. We use the model as a **judge** with a rubric and a structured verdict. As in the tutorial: the judge can be wrong, so spot-check it by hand, and a different model as the judge reduces the risk of the model agreeing with itself.

In [25]:
class Verdict(BaseModel):
    follows_twist: bool = Field(description="True only if the room clearly satisfies the twist.")
    reason: str = Field(description="One sentence explaining the verdict.")

JUDGE_RUBRIC = '''You are grading a room from the Mimic Logic puzzle generator.
Normal chests tell the truth and Mimics lie. You are given the design brief's twist,
the grid layout, every chest's statement, and the answer.
Mark follows_twist = true only if the room clearly satisfies the twist as written.
If the twist is only partly satisfied, mark it false.'''

def judge_record(record):
    room = record["room"]
    content = (f"Twist: {room['meta']['brief']['twist']}\n\n"
               f"{render(room)}\n\nAnswer: {room['solution']}")
    return judge.invoke([{"role": "system", "content": JUDGE_RUBRIC},
                         {"role": "user", "content": content}])

if USE_OFFLINE_MODEL:
    print("The judge needs a real model; skipped with the offline stand-in.")
else:
    judge = model.with_structured_output(Verdict)
    for r in baseline_records:
        if r["room"]["meta"]["source"] != "agent":
            print(f"{r['id']:<17}{'(fallback)':<14}")
            continue
        verdict = judge_record(r)
        r["judge"] = verdict
        print(f"{r['id']:<17}{'follows' if verdict.follows_twist else 'does NOT':<14}{verdict.reason}")

level1_a         follows       The solution relies on the mimic-count hint (there is exactly 1 mimic), which leads to identifying B as the sole mimic, satisfying the twist requirement.
level2_a         follows       The two mimics, A and B, are placed far apart on the grid, satisfying the twist that two chests of the same kind should be far apart.
level2_b         follows       Chest A is a corner chest and is the sole mimic, making it the key to the room's solution as required by the twist.
level3_a         follows       Chest A is normal but makes a statement that sounds suspicious, fulfilling the twist that the normal chest looks most suspicious at first glance.
level4_a         follows       No chest directly calls another chest a mimic; all statements are conditional or about counts, not direct accusations.
level4_b         follows       Chest B's conditional statement 'If Chest D is a mimic, then Chest F is a mimic' has a false 'if' part since D is normal, fulfilling the twist re

## Step 4: Error analysis

Pick a failing case and read its trace. Where did it go wrong: the plan (wrong mimic count), a statement that broke the intended answer, or a revision that never resolved the ambiguity?

In [26]:
failing = [r for r in baseline_records if not all(r["checks"].values())]
if failing:
    r = failing[0]
    print(f"Case {r['id']}: {r['note']}\n")
    print_trace(r["steps"])
else:
    print("Every case passed every check.")

Every case passed every check.


# 7. Your Turn

Ideas for improving the agent, **one change at a time, measured with the evaluation above**:

1. **Policy.** If the agent often misses the brief's mimic count, add a rule to the system prompt: *"Before check_room, count your mimics against the brief."* Re-run Section 6.
2. **A stricter gate.** `submit_room` currently accepts a unique room even if it breaks the brief. Make it reject rooms with `brief_problems` and measure the effect on **brief** and on **delivered**: a stricter gate may mean more fallbacks.
3. **Temperature.** Compare 0.3 and 0.9: does more randomness improve variety without hurting **delivered**?
4. **Variety.** Generate 10 rooms at the same level and count distinct layouts and statement-type mixes.